<a href="https://colab.research.google.com/github/yiyu-chen-labs/llm-from-scratch/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Baseline 探索筆記 — JQaRA (純文字檢索)

## 目標
先用純文字資料(JQaRA)熟悉 RAG 檢索的整條 workflow
專注在「檢索」這一半:向量化 → 相似度排序 → 用 label 驗證準不準

## 資料:JQaRA (hotchpotch/JQaRA)
- 日文 RAG 評估資料集,知識庫是 Wikipedia 段落(純文字,無圖無 PDF)
- 結構:資料是"flat"的,一題的每個候選段落各佔一列,用"q_id"binding
- 關鍵欄位:question(問題)、text(候選段落)、answers(正解)、
  label(1=正解段落 / 0=干擾項)、q_id(綁定同題的鑰匙)
- label 是"評估用的答案卡",系統實際檢索時看不到它

## 做了什麼
1. 用 q_id 篩出一題 + 它的所有候選段落
2. embedding 模型:intfloat/multilingual-e5-small(384 維,支援日文)
3. 把 question 和候選段落向量化,算 cosine similarity,由高到低排序
4. 用 label 驗證:相似度排名前面的,是不是 label=1 的正解

## 關鍵觀察
- 檢索**大致有效**:正解(label=1)被撈進前 3 名(排第 2、第 3)
- 但**排序不完美**:第 1 名是一個 label=0 的干擾段落(海蛞蝓/ウミウシ,
  相似度 0.690),它擠掉了正解
- **原因**:問題問「海の天使」(クリオネ),干擾段落提到「海のナメクジ」,
  兩者**語意極相近**,embedding 因此誤判它最相關
- **結論**:embedding 找的是「語意最相似」,不等於「含正解」
  **相似 ≠ 正確** — 這是 RAG 的核心難題,我親眼看到它發生

## 這對 Phase 2 的意義
- **Re-ranking (Week 11)**:baseline 把相關段落撈進來了造成排序瑕疵
  (干擾項排第一)→ 需要更精細的模型重排 and this is why we need re-ranking
- **引用忠實度 / abstention (Week 12)**:若 LLM 拿到第一名那個無關段落作答,可能答錯 → 需要檢查「答案是否真被段落支持」的機制

## 待辦 / 下一步
- [ ] 玩法 B:自己建 FAISS 向量庫,從整庫檢索(非現成候選)。
- [ ] 把「正解平均排第幾」算成量化指標(MRR / recall@k)當評估基準。
- [ ] chunking 練習需另找「原始長文件」資料集(JQaRA 段落已切好,練不到 chunking)。
- [ ] 生成那半(LLM 讀 prompt 作答)待接 API 後做。

In [18]:
from datasets import load_dataset

ds = load_dataset("hotchpotch/JQaRA")
print(ds)          # 看有哪些 split、多少筆、欄位有哪些

DatasetDict({
    unused: Dataset({
        features: ['id', 'q_id', 'passage_row_id', 'label', 'text', 'title', 'question', 'answers'],
        num_rows: 24900
    })
    dev: Dataset({
        features: ['id', 'q_id', 'passage_row_id', 'label', 'text', 'title', 'question', 'answers'],
        num_rows: 86850
    })
    test: Dataset({
        features: ['id', 'q_id', 'passage_row_id', 'label', 'text', 'title', 'question', 'answers'],
        num_rows: 166700
    })
})


In [19]:
sample = ds['dev'][0]
print(sample)

{'id': 'QA20CAPR-0001#3777755', 'q_id': 'QA20CAPR-0001', 'passage_row_id': '3777755', 'label': 1, 'text': 'トヨオカハダカカメガイ(豊岡裸亀貝)、学名 Pneumodermopsis ciliata は、ニュウモデルマ科に分類される浮遊性の腹足類(=広義の巻貝)の一種。体長数mmから最大で15mmほどで、比較的近縁なクリオネに似た姿の海洋プランクトンである。北半球のいくつかの海域に分布し、他の浮遊性の貝類や甲殻類を捕食する。Pneumodermopsis 属のタイプ種。和名は日本で最初に報告された標本が兵庫県豊岡市で採取されたものであったことに因む。', 'title': 'トヨオカハダカカメガイ', 'question': '和名をハダカカメガイといい、実は巻き貝の一種とされている、その姿から「流氷の天使」と呼ばれる動物は何でしょう?', 'answers': ['クリオネ']}


In [20]:
sample = ds['dev'][0]
for key, value in sample.items():
    print(f"{key}: {value}")
    print("-" * 40)

id: QA20CAPR-0001#3777755
----------------------------------------
q_id: QA20CAPR-0001
----------------------------------------
passage_row_id: 3777755
----------------------------------------
label: 1
----------------------------------------
text: トヨオカハダカカメガイ(豊岡裸亀貝)、学名 Pneumodermopsis ciliata は、ニュウモデルマ科に分類される浮遊性の腹足類(=広義の巻貝)の一種。体長数mmから最大で15mmほどで、比較的近縁なクリオネに似た姿の海洋プランクトンである。北半球のいくつかの海域に分布し、他の浮遊性の貝類や甲殻類を捕食する。Pneumodermopsis 属のタイプ種。和名は日本で最初に報告された標本が兵庫県豊岡市で採取されたものであったことに因む。
----------------------------------------
title: トヨオカハダカカメガイ
----------------------------------------
question: 和名をハダカカメガイといい、実は巻き貝の一種とされている、その姿から「流氷の天使」と呼ばれる動物は何でしょう?
----------------------------------------
answers: ['クリオネ']
----------------------------------------


In [21]:
from pprint import pprint
pprint(sample)

{'answers': ['クリオネ'],
 'id': 'QA20CAPR-0001#3777755',
 'label': 1,
 'passage_row_id': '3777755',
 'q_id': 'QA20CAPR-0001',
 'question': '和名をハダカカメガイといい、実は巻き貝の一種とされている、その姿から「流氷の天使」と呼ばれる動物は何でしょう?',
 'text': 'トヨオカハダカカメガイ(豊岡裸亀貝)、学名 Pneumodermopsis ciliata '
         'は、ニュウモデルマ科に分類される浮遊性の腹足類(=広義の巻貝)の一種。体長数mmから最大で15mmほどで、比較的近縁なクリオネに似た姿の海洋プランクトンである。北半球のいくつかの海域に分布し、他の浮遊性の貝類や甲殻類を捕食する。Pneumodermopsis '
         '属のタイプ種。和名は日本で最初に報告された標本が兵庫県豊岡市で採取されたものであったことに因む。',
 'title': 'トヨオカハダカカメガイ'}


In [22]:
import pandas as pd

df = ds['dev'].to_pandas()

print("總筆數", len(df))
print("不同問題數", df['q_id'].nunique())

first_qid = df['q_id'].iloc[0]
one_question = df[df['q_id']==first_qid]


print("\n這一題的 q_id:", first_qid)
print("這一題有幾個候選段落:", len(one_question))
print("其中 label=1（正解）有幾個:", (one_question['label'] == 1).sum())


總筆數 86850
不同問題數 1737

這一題的 q_id: QA20CAPR-0001
這一題有幾個候選段落: 50
其中 label=1（正解）有幾個: 4


In [23]:
first_qid = df['q_id'].iloc[0]

print("篩選前，df 總列數:", len(df))          # 幾萬
print("first_qid 是:", first_qid)

one_question = df[df['q_id'] == first_qid]
print("篩選後，one_question 列數:", len(one_question))  # 約100（只有這題）
print("篩選後，df 還是:", len(df), "列（沒變）")        # 還是幾萬，證明 df 沒被改

篩選前，df 總列數: 86850
first_qid 是: QA20CAPR-0001
篩選後，one_question 列數: 50
篩選後，df 還是: 86850 列（沒變）


In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("Model is Ready")
print("向量維度",model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model is Ready
向量維度 384


/tmp/ipykernel_3171/4284073539.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("向量維度",model.get_sentence_embedding_dimension())


In [25]:
question = one_question['question'].iloc[0]

passages = one_question['text'].tolist()
print("Question:",question)
print("Answer:",passages)

Question: 和名をハダカカメガイといい、実は巻き貝の一種とされている、その姿から「流氷の天使」と呼ばれる動物は何でしょう?
Answer: ['トヨオカハダカカメガイ(豊岡裸亀貝)、学名 Pneumodermopsis ciliata は、ニュウモデルマ科に分類される浮遊性の腹足類(=広義の巻貝)の一種。体長数mmから最大で15mmほどで、比較的近縁なクリオネに似た姿の海洋プランクトンである。北半球のいくつかの海域に分布し、他の浮遊性の貝類や甲殻類を捕食する。Pneumodermopsis 属のタイプ種。和名は日本で最初に報告された標本が兵庫県豊岡市で採取されたものであったことに因む。', 'ハダカカメガイは北極海に分布。暖かい水域に分布する種はハダカカメガイよりはるかに小さい。有殻翼足類のみを捕食する種も何種かいる。これらの種は、軟体動物に共通の歯舌を備えた口をもち、頭足類の吸盤に似た触手で獲物をつかむことができる。さらに、側足と呼ばれる「翼」で、より大きな「翼」を持つ有殻翼足亜目よりも非常に速く泳ぐことができる。これらの種以外のほとんどの種は、動物プランクトンを常食にする。南極海に分布するクリオネの一種 Clione antarctica は、微量の忌避物質 pteroenone を合成することで捕食者から身を守っている。同種の局所的な存在密度は異常であり、最大で1mあたり300匹になることもある。捕食者が C. antarctica を食べないので、端脚目等その中に住む動物もいる。', '当時は、クリオネは深海にいると思われていて、渚でも採取できるということで話題となったりしている。水族館は1993年には、貝殻を持たない巻き貝の一種、クリオネ(和名ハダカカメガイ)の飼育に世界で初めて成功し、1993年4月28日から、「流氷の天使」のキャッチコピーを付けて日本初の展示を行う。館長の本間保が“流氷の天使”を命名した。水族館はクリオネについて「暖かくなり、流氷が北に去るとともに姿を消す謎の多い不思議な生物」だと紹介している。水族館はクリオネの通年展示を実現したため、一躍有名となり、入館者数も増加し、また、各方面から注目を集める。人気にあやかって、1995年7月26日、水族館と網走刑務所は共同で同所製の「クリオネグッズ」を売り出している。', '巻貝の仲間であ

In [26]:
q_vec = model.encode(question)
p_vecs = model.encode(passages)

print("\n問題向量的形狀:", q_vec.shape)   # (384,) → 一串 384 個數字
print("段落向量的形狀:", p_vecs.shape)    # (N, 384) → N 段，每段 384
print("\n問題向量前 5 個數字:", q_vec[:5]) # 偷看一下「文字變成的數字」長怎樣


問題向量的形狀: (384,)
段落向量的形狀: (50, 384)

問題向量前 5 個數字: [-0.03941223  0.00033276  0.129626    0.13468935 -0.21503472]


In [27]:
from sentence_transformers import util
import numpy as np


#cosine similarity
scores = util.cos_sim(q_vec, p_vecs)[0]
ranked_idx = np.argsort(scores.numpy())[::-1]


print("排名 | 相似度 | label | 段落前30字")
print("-" * 60)
for rank, i in enumerate(ranked_idx[:10]):
    label = one_question['label'].iloc[i]
    text_preview = passages[i][:30]
    mark = "✓正解" if label == 1 else ""
    print(f"{rank+1:>3}  | {scores[i]:.3f} | {label} {mark} | {text_preview}")

排名 | 相似度 | label | 段落前30字
------------------------------------------------------------
  1  | 0.690 | 0  | よくウミウシ類は「海のナメクジ」とも呼称されるが、その仲間と
  2  | 0.649 | 1 ✓正解 | 当時は、クリオネは深海にいると思われていて、渚でも採取できる
  3  | 0.648 | 1 ✓正解 | 巻貝の仲間であるが、成長すると完全に貝殻を失う(裸殻翼足類共
  4  | 0.587 | 0  | 体長50-80mmほどで、サクラエビよりも大きく、やや左右に
  5  | 0.542 | 0  | オカミミガイ(陸耳貝)、学名 Ellobium chinen
  6  | 0.534 | 0  | 虫卵は宿主の排泄物とともに外界に出てミラシジウムが孵化する。
  7  | 0.524 | 0  | アマンユベニガイ(奄美世紅貝、学名 Pharaonella 
  8  | 0.522 | 0  | カワアカメ(川赤目、英:Barbel chub 、中:赤眼鳟
  9  | 0.513 | 0  | オビクロモンサカタザメ(Southern_shovelnos
 10  | 0.505 | 0  | ヒラムシ(扁虫、平虫)は、磯の石の下にすむ扁形動物渦虫綱ヒラ


In [28]:
# 找出所有正解(label==1)排在第幾名
for rank, i in enumerate(ranked_idx):
    if one_question['label'].iloc[i] == 1:
        print(f"正解排在第 {rank+1} 名 (相似度 {scores[i]:.3f})")

正解排在第 2 名 (相似度 0.649)
正解排在第 3 名 (相似度 0.648)
正解排在第 14 名 (相似度 0.481)
正解排在第 38 名 (相似度 0.356)


In [29]:
#----以上只是對ranked一題now i am goin to ranked all of the queistons

def evaluate_one_question(qid):
  one_qid = df[df['q_id'] == qid] #extract each q_id

  #2.取問題＋候選段落
  question = one_question['question'].iloc[0]
  passages = one_question['text'].tolist()
  labels = one_question['lables'].tolist()

  #3.向量化＋相似度＋排序
  q_vec = model.encode(question)
  p_vecs = model.encode(passages)
  ranked = util.cos_sim(q_vec, p_vecs)[0] #similary callcaued with cos




In [30]:
def evaluate_one_question(qid):
    """給一個 q_id，回傳這題的正解排在第幾名（rank，從1開始）的清單。"""
    # 1. 篩出這一題
    one_q = df[df['q_id'] == qid]
    unique_q = df['q_id'].unique()
    # 2. 取問題 + 候選段落
    question = one_q['question'].iloc[0]
    passages = one_q['text'].tolist()
    labels = one_q['label'].tolist()      # 每段的 label，順序跟 passages 對齊

    # 3. 向量化 + 相似度 + 排序
    q_vec = model.encode(question)
    p_vecs = model.encode(passages)
    scores = util.cos_sim(q_vec, p_vecs)[0]
    ranked_idx = np.argsort(scores.numpy())[::-1]

    # 4. 找出正解(label==1)排在第幾名
    gold_ranks = []
    for rank, i in enumerate(ranked_idx):
        if labels[i] == 1:
            gold_ranks.append(rank + 1)   # +1 因為 rank 從0開始，人看的名次從1開始

    return gold_ranks


# 測試：用昨天那題跑跑看
test_qid = df['q_id'].iloc[0]
print("q_id:", test_qid)
print("正解排名:", evaluate_one_question(test_qid))

q_id: QA20CAPR-0001
正解排名: [2, 3, 14, 38]


In [38]:
unique_q = df['q_id'].unique()
hits = []                              # 複數，裝所有題的結果
for qid in unique_q[:100]:              # 記得取前20題測試，不然跑全部很慢
    gold_rank = evaluate_one_question(qid)
    best_rank = min(gold_rank)
    hit = best_rank <= 5               # 單數，這一題的結果
    hits.append(hit)                   # 存進清單

recall_at_5 = sum(hits) / len(hits)    # 平均
print("recall@5:", recall_at_5)

recall@5: 0.89


In [39]:
unique_q = df['q_id'].unique()
hits = []                              # 複數，裝所有題的結果
for qid in unique_q[:100]:              # 記得取前20題測試，不然跑全部很慢
    gold_rank = evaluate_one_question(qid)
    best_rank = min(gold_rank)
    hit = best_rank <= 10               # 單數，這一題的結果
    hits.append(hit)                   # 存進清單

recall_at_10= sum(hits) / len(hits)    # 平均
print("recall@10:", recall_at_10)

recall@10: 0.92


In [40]:
unique_q = df['q_id'].unique()
hits = []                              # 複數，裝所有題的結果
for qid in unique_q[:100]:              # 記得取前20題測試，不然跑全部很慢
    gold_rank = evaluate_one_question(qid)
    best_rank = min(gold_rank)
    hit = best_rank <= 1               # 單數，這一題的結果
    hits.append(hit)                   # 存進清單

recall_at_1 = sum(hits) / len(hits)    # 平均
print("recall@1:", recall_at_1)

recall@1: 0.6
